# ⚽ WC2026 Actor Suite — Colab Test Notebook

Test both actors **locally in Colab** before deploying to Apify.

**Workflow:**
1. Clone repo from GitHub
2. Install dependencies
3. Test Fixture Scraper
4. Test Match Predictor
5. Push changes back to GitHub → Apify auto-deploys

In [ ]:
# ── Step 1: Clone your GitHub repo ─────────────────────────────────────────
# Replace YOUR_USERNAME with your actual GitHub username

!git clone https://github.com/YOUR_USERNAME/wc2026-actors.git
%cd wc2026-actors
!ls

In [ ]:
# ── Step 2: Install dependencies ────────────────────────────────────────────
!pip install requests beautifulsoup4 apify-client -q

In [ ]:
# ── Step 3: Test Fixture Scraper ─────────────────────────────────────────────
import sys
sys.path.insert(0, 'actors/wc2026-fixtures')

# Import and run local test
import importlib.util, sys
spec = importlib.util.spec_from_file_location('fixtures', 'actors/wc2026-fixtures/main.py')
fixtures_mod = importlib.util.load_module_from_spec(spec)
spec.loader.exec_module(fixtures_mod)

# Run local test
fixtures_mod._run_local(input_data={})

In [ ]:
# ── Step 4: Test Match Predictor ─────────────────────────────────────────────
spec2 = importlib.util.spec_from_file_location('predictor', 'actors/wc2026-predictor/main.py')
pred_mod = importlib.util.load_module_from_spec(spec2)
spec2.loader.exec_module(pred_mod)

pred_mod._run_local()

In [ ]:
# ── Step 5: Run a custom prediction ──────────────────────────────────────────
import json

# Change these to any WC2026 teams
HOME = 'Japan'
AWAY = 'Netherlands'
LANG = 'zh-CN'   # en | zh-CN | zh-TW

result = pred_mod._predict(HOME, AWAY, 'group_stage', LANG, False, '')

print(f"\n{'='*50}")
print(f"  {HOME} vs {AWAY}")
print(f"{'='*50}")
print(f"  {HOME} Win:  {result['home_win_pct']}%")
print(f"  Draw:        {result['draw_pct']}%")
print(f"  {AWAY} Win: {result['away_win_pct']}%")
print(f"  ─────────────────────────────")
print(f"  Prediction:  {result['predicted_winner']}")
print(f"  AH Pick:     {result['asian_handicap_pick']} ({result['asian_handicap_line']:+.1f})")
print(f"  Confidence:  {result['confidence']}/10")
print(f"  Elo:         {HOME} {result['home_elo']} vs {AWAY} {result['away_elo']}")

In [ ]:
# ── Step 6: Filter fixtures by team ──────────────────────────────────────────
team_filter = 'Japan'

japan_fixtures = [
    f for f in fixtures_mod.CONFIRMED_FIXTURES
    if team_filter.lower() in f['home_team'].lower()
    or team_filter.lower() in f['away_team'].lower()
]

print(f"\n{team_filter} fixtures in WC2026:")
for f in japan_fixtures:
    print(f"  Group {f['group']} | {f['date']} {f['kickoff_local']} {f['timezone']}")
    print(f"  {f['home_team']} vs {f['away_team']} — {f['city']}")
    print()

In [ ]:
# ── Step 7: Push changes to GitHub ───────────────────────────────────────────
# Run this after making any edits — Apify will auto-rebuild

!git config user.email 'you@email.com'
!git config user.name 'Your Name'
!git add .
!git commit -m 'update: [describe your change]'
# !git push   # Uncomment when ready to deploy